In [1]:
# Import necessary libraries
from transformers import MarianMTModel, MarianTokenizer
import gradio as gr

# Define the model name
model_name = "Helsinki-NLP/opus-mt-en-de"

# Load tokenizer and model (no token needed)
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate(text):
    # Tokenize input text
    inputs = tokenizer(text, return_tensors="pt", padding=True)

    # Generate translation
    translated = model.generate(**inputs)

    # Decode translated text
    output = tokenizer.decode(translated[0], skip_special_tokens=True)

    return output

# Gradio interface
iface = gr.Interface(
    fn=translate,
    inputs=gr.Textbox(label="Enter English Text"),
    outputs=gr.Textbox(label="German Translation"),
    title="English → German Translator",
    description="Type English text and get German translation."
)

# Launch interface
iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d82a8abbaaefe5df98.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
!pip install transformers sentencepiece googletrans==4.0.0-rc1 flask flask-ngrok

In [8]:
!pip install flask-ngrok pyngrok sacremoses


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.1 MB/s eta 0:00:00


In [9]:
%%writefile app.py

from flask import Flask, render_template, request
from flask_ngrok import run_with_ngrok
from transformers import MarianMTModel, MarianTokenizer
from googletrans import Translator
from difflib import SequenceMatcher

app = Flask(__name__)
run_with_ngrok(app)

model_name = "Helsinki-NLP/opus-mt-en-de"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

translator = Translator()

def hf_translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

def google_translate(text):
    return translator.translate(text, dest="de").text

def similarity(a,b):
    return SequenceMatcher(None,a,b).ratio()

@app.route("/", methods=["GET","POST"])

def home():

    hf=""
    google=""
    score=0
    flag=False

    if request.method=="POST":

        text=request.form["text"]

        hf=hf_translate(text)
        google=google_translate(text)

        score=similarity(hf,google)

        if score<0.75:
            flag=True

    return render_template("index.html",
                           hf=hf,
                           google=google,
                           score=round(score*100,2),
                           flag=flag)

if __name__=="__main__":
    app.run()

Overwriting app.py


In [3]:
!mkdir templates

In [6]:
%%writefile templates/index.html
<!DOCTYPE html>
<html>

<head>

<title>AI Translator Pro</title>

<style>

body{
font-family: Arial;
background: linear-gradient(120deg,#0f2027,#203a43,#2c5364);
color:white;
text-align:center;
padding:40px;
}

.container{
background:white;
color:black;
padding:30px;
border-radius:10px;
width:60%;
margin:auto;
box-shadow:0px 10px 30px rgba(0,0,0,0.4);
}

textarea{
width:90%;
height:120px;
padding:10px;
font-size:16px;
border-radius:6px;
border:1px solid gray;
}

button{
background:#007bff;
color:white;
padding:12px 30px;
border:none;
border-radius:6px;
font-size:18px;
cursor:pointer;
margin-top:10px;
}

button:hover{
background:#0056b3;
}

.result{
margin-top:20px;
padding:15px;
background:#f4f4f4;
border-radius:6px;
}

.flag{
font-size:28px;
}

.warning{
color:red;
font-weight:bold;
}

</style>

</head>

<body>

<h1>🌍 AI Translator Pro</h1>

<div class="container">

<form method="post">

<textarea name="text" placeholder="Enter English text"></textarea>

<br>

<button type="submit">Translate</button>

</form>

{% if hf %}

<div class="result">

<h3 class="flag">🤖 AI Translation 🇩🇪</h3>

<p>{{hf}}</p>

</div>

<div class="result">

<h3 class="flag">🌐 Google Translation 🇩🇪</h3>

<p>{{google}}</p>

</div>

<div class="result">

<h3>Accuracy Score</h3>

<p>{{score}} %</p>

{% if flag %}

<p class="warning">🚩 Translation may be inaccurate</p>

{% else %}

<p style="color:green">✅ Translation looks accurate</p>

{% endif %}

</div>

{% endif %}

</div>

</body>

</html>

Overwriting templates/index.html


In [11]:
!pip install flask transformers torch deep-translator sacremoses pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.2 MB/s eta 0:00:00


In [14]:
from pyngrok import ngrok
ngrok.set_auth_token("3BmoFLcyBM24fGQ1gzQ2tEqPcu0_4AE9YEtMLbehRCQQAHL8K")

In [ ]:
from flask import Flask, render_template_string, request, jsonify
from transformers import MarianMTModel, MarianTokenizer
from deep_translator import GoogleTranslator
from pyngrok import ngrok

app = Flask(__name__)

# Load AI Translation Model
model_name = "Helsinki-NLP/opus-mt-en-de"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def ai_translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

def google_translate(text):
    return GoogleTranslator(source='en', target='de').translate(text)

# HTML UI
HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
<title>AI Translator</title>

<style>

body{
font-family:Arial;
background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);
color:white;
text-align:center;
padding:40px;
}

.container{
background:rgba(255,255,255,0.1);
padding:40px;
border-radius:15px;
width:600px;
margin:auto;
}

textarea{
width:100%;
height:120px;
border-radius:10px;
padding:10px;
font-size:16px;
}

button{
background:#00c6ff;
border:none;
padding:12px 25px;
font-size:16px;
border-radius:8px;
cursor:pointer;
margin-top:15px;
}

.result{
margin-top:20px;
font-size:18px;
}

.flag{
font-size:24px;
}

.warning{
color:#ff4c4c;
font-weight:bold;
}

</style>
</head>

<body>

<h1>🌍 AI Translator</h1>

<div class="container">

<p>🇺🇸 English → 🇩🇪 German</p>

<textarea id="text" placeholder="Enter English text"></textarea>

<br>

<button onclick="translateText()">Translate</button>

<div class="result">

<p><b>AI Translation:</b></p>
<p id="ai"></p>

<p><b>Google Translation:</b></p>
<p id="google"></p>

<p id="warning" class="warning"></p>

</div>

</div>

<script>

function translateText(){

let text=document.getElementById("text").value

fetch("/translate",{
method:"POST",
headers:{
"Content-Type":"application/json"
},
body:JSON.stringify({text:text})
})
.then(res=>res.json())
.then(data=>{

document.getElementById("ai").innerText=data.ai
document.getElementById("google").innerText=data.google

if(data.flag){
document.getElementById("warning").innerText="⚠ Translation may be inaccurate"
}
else{
document.getElementById("warning").innerText=""
}

})

}

</script>

</body>
</html>
"""

@app.route("/")
def home():
    return render_template_string(HTML_PAGE)

@app.route("/translate", methods=["POST"])
def translate():
    text = request.json["text"]

    ai = ai_translate(text)
    google = google_translate(text)

    flag = ai.lower() != google.lower()

    return jsonify({
        "ai": ai,
        "google": google,
        "flag": flag
    })

# Run server
public_url = ngrok.connect(5000)
print("Public URL:", public_url)

app.run(port=5000)

Public URL: NgrokTunnel: "https://salvational-unringing-moon.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:31:58] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:31:59] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:32:07] "POST /translate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:32:09] "POST /translate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:32:56] "POST /translate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:32:57] "POST /translate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:32:57] "POST /translate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Apr/2026 04:33:31] "POST /translate HTTP/1.1" 200 -
